# SAR → EO — Local Training

**For your own machine or the college GPU. Not Kaggle.**

If you are on Kaggle, use `kaggle_phase1_train.ipynb` instead — that one expects
`/kaggle/input` and `/kaggle/working`, which don't exist here.

---

## What this notebook does

Turns radar images into colour satellite images. You give it a grey Sentinel-1
SAR picture; it produces the Sentinel-2 optical picture that *would* have been
taken if there were no clouds.

Run the cells top to bottom. Every cell explains what it's doing and what
"working" looks like, so you can tell the difference between a real problem and
normal output.

## Before you start

1. **A CUDA GPU with 16 GB or more.** The A5000's 24 GB is comfortable.
2. **Dependencies installed** — `pip install -r requirements.txt`
3. **The dataset downloaded** into `data/sentinel12/` (cell 3 checks this and
   tells you exactly what's wrong if it isn't right)
4. **Jupyter started from the repo folder**, so that `train.py` sits next to
   this notebook

## How the session limit works

You are not expected to sit here for 20 hours. Set `SESSION_MINUTES` in cell 7
to however long you have the machine. Training stops before it overruns, saves
everything, and picks up exactly where it left off next time you run it.

Nothing is lost between sessions — not the optimiser, not the learning rate
schedule, not your best model so far.

## 1 · Where am I, and is everything here?

This checks you started Jupyter in the right folder. If it fails, close Jupyter,
`cd` into the `sar2eo` folder, and run `jupyter notebook` from there.

In [ ]:
import os, sys, glob, json, shutil, subprocess

print("Working directory:")
print("   ", os.getcwd())
print()

REQUIRED = ["train.py", "config.yaml", "run_ablations.py", "eval.py",
            "data/dataloader.py", "models/generator.py"]
missing = [f for f in REQUIRED if not os.path.exists(f)]

if missing:
    print("PROBLEM — these files are not here:")
    for f in missing:
        print("   ", f)
    print()
    print("You started Jupyter in the wrong folder. Close it, then:")
    print("    cd path/to/sar2eo")
    print("    jupyter notebook")
    raise SystemExit("wrong directory")

print("All project files found. You are in the right place.")

## 2 · Do I have a working GPU?

Training on a CPU would take weeks, so this cell refuses to continue without a
GPU. `torch.cuda.is_available()` being `False` almost always means PyTorch was
installed without CUDA support — reinstall it from pytorch.org, choosing your
CUDA version.

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    print()
    print("PROBLEM — no GPU visible to PyTorch.")
    print("Check `nvidia-smi` works in a terminal. If it does, PyTorch was")
    print("installed without CUDA. Reinstall from https://pytorch.org/")
    raise SystemExit("no GPU")

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU             : {GPU_NAME}")
print(f"VRAM            : {VRAM_GB:.1f} GB")
print(f"CPU cores       : {os.cpu_count()}")

# batch_size is the biggest VRAM lever. These are safe starting points.
if   VRAM_GB >= 22: BATCH_SIZE = 16
elif VRAM_GB >= 15: BATCH_SIZE = 8
else:               BATCH_SIZE = 4

# Workers decode and augment images on the CPU while the GPU computes. Too few
# and the GPU waits; too many and they fight over cores.
NUM_WORKERS = max(2, min(8, (os.cpu_count() or 4) // 2))

print()
print(f"Chosen batch_size  : {BATCH_SIZE}")
print(f"Chosen num_workers : {NUM_WORKERS}")
print("(Lower BATCH_SIZE if you hit an out-of-memory error later.)")

## 3 · Is the dataset where it needs to be?

The loader expects this layout, one folder per terrain type:

```
data/sentinel12/
├── agri/
│   ├── s1/      ← the grey SAR images
│   └── s2/      ← the matching colour optical images
├── barrenland/
│   ├── s1/
│   └── s2/
├── grassland/
│   └── ...
└── urban/
    └── ...
```

A file in `s1/` and its partner in `s2/` **must have the same filename**. That is
how they get paired.

If your download unzipped into an extra nested folder — something like
`data/sentinel12/v_2/agri/` — this cell finds it and tells you the path to use.

In [ ]:
from pathlib import Path

TERRAIN_KEYS = {"agri", "urban", "grassland", "barrenland",
                "forest", "water", "mountain"}

DATA_ROOT = None
search_from = Path("data")

if not search_from.exists():
    print("PROBLEM — there is no 'data' folder. Create it and put the dataset in.")
    raise SystemExit("no data folder")

# Walk down looking for a folder containing 2+ terrain subfolders.
for dirpath, dirnames, _ in os.walk(search_from):
    found = {d.lower() for d in dirnames} & TERRAIN_KEYS
    if len(found) >= 2:
        DATA_ROOT = dirpath.replace("\\", "/")
        print(f"Dataset found at : {DATA_ROOT}")
        print(f"Terrain folders  : {sorted(found)}")
        break

if DATA_ROOT is None:
    print("PROBLEM — could not find terrain folders under data/.")
    print()
    print("Here is what IS under data/ (3 levels deep):")
    for dirpath, dirnames, filenames in os.walk(search_from):
        depth = dirpath.replace(str(search_from), "").count(os.sep)
        if depth > 2:
            continue
        print("   " + "   " * depth + os.path.basename(dirpath) + "/"
              + (f"   ({len(filenames)} files)" if filenames else ""))
    print()
    print("You need at least two of:", sorted(TERRAIN_KEYS))
    raise SystemExit("dataset not found")

### Now check the pairs actually match up

Counting files isn't enough — an `s1` image is only usable if a file with the
*same name* exists in `s2`. This counts real pairs and shows you some filenames,
which matters for the next cell.

### One warning you *will* see, and should ignore

Later cells print this, once per terrain:

```
[WARNING] Pairing 'v_2/agri' by SORTED INDEX (no filename match)
```

That is expected for this dataset and is **not** a problem. The SAR and optical
files are named `..._s1_59_p10.png` and `..._s2_59_p10.png` — identical except
for `s1`/`s2` — so exact-name matching finds nothing and the loader pairs them
by sorted position instead.

All 16,000 pairs were checked against this archive and **none are mispaired**.
The warning exists because sorted-position pairing *can* go wrong on other
datasets, so it says so rather than staying silent. Read the sample pairs it
prints; if each line shows the same numbers on both sides, it is correct.

In [ ]:
total_pairs = 0
sample_names = []

print(f"{'terrain':<14}{'s1':>8}{'s2':>8}{'matched':>10}")
print("-" * 40)

for tdir in sorted(Path(DATA_ROOT).iterdir()):
    if not tdir.is_dir():
        continue
    s1d, s2d = tdir / "s1", tdir / "s2"
    if not (s1d.is_dir() and s2d.is_dir()):
        print(f"{tdir.name:<14}   no s1/ or s2/ subfolder - skipped")
        continue
    s1 = {p.name for p in s1d.glob("*.png")}
    s2 = {p.name for p in s2d.glob("*.png")}
    matched = s1 & s2
    total_pairs += len(matched)
    if not sample_names and matched:
        sample_names = sorted(matched)[:3]
    print(f"{tdir.name:<14}{len(s1):>8,}{len(s2):>8,}{len(matched):>10,}")

print("-" * 40)
print(f"{'TOTAL':<14}{'':>8}{'':>8}{total_pairs:>10,}")
print()

if total_pairs == 0:
    print("PROBLEM — zero matched pairs.")
    print("Either the folders are empty, or s1/ and s2/ filenames differ.")
    raise SystemExit("no pairs")

print(f"{total_pairs:,} usable pairs. Expect roughly 16,000 for this dataset.")
print()
print("Example filenames:")
for n in sample_names:
    print("   ", n)

## 4 · The most important check in this notebook

Read this bit — it is the thing most likely to quietly ruin your results.

The dataset was made by sliding a window across big satellite scenes and cutting
out overlapping tiles. So `..._p265.png` and `..._p266.png` are **the same patch
of ground, shifted slightly**.

If those two tiles land on opposite sides of the train/test split, the model
gets tested on ground it already memorised. Your scores come out high and mean
nothing.

To avoid that, tiles are grouped by the **scene** they were cut from, and whole
scenes go to one side of the split. That needs the scene ID readable from the
filename — like `ROIs1970_fall_s1_13_p265.png`, where `13` is the scene.

This cell tells you whether that's working on your files.

In [ ]:
sys.path.insert(0, os.getcwd())
from data.dataloader import _scene_key, _SCENE_RE

print("filename  ->  scene group")
print("-" * 64)
parsed = 0
for n in sample_names:
    key = _scene_key(f"{DATA_ROOT}/agri/s1/{n}")
    ok  = bool(_SCENE_RE.search(n))
    parsed += ok
    print(f"{n[:34]:<36} -> {key[-26:]}   {'OK' if ok else 'NOT PARSED'}")
print()

if parsed == len(sample_names) and sample_names:
    print("GOOD — scene IDs are readable. Splits will be properly separated.")
    print("The next cell should report ~30 scenes for this dataset.")
else:
    print("WARNING — the scene ID could not be read from these filenames.")
    print("Grouping falls back to whole folders, so the split becomes")
    print("terrain-based: much harder, and not what the config claims.")
    print()
    print("It is still leak-free, so it is safe to train. But tell Claude the")
    print("filenames above before you spend GPU hours - the pattern is a")
    print("small fix.")

## 5 · Build the configuration

Everything the training run needs, written to `config_local.yaml`.

**The one number to change is `SESSION_MINUTES`** — set it to how long you
actually have the machine. 225 means "stop before 3 h 45 m", which leaves
15 minutes of a 4-hour slot for setup and copying files off.

In [ ]:
import yaml

# ---- The settings you might actually want to change ---------------------
SESSION_MINUTES = 225     # how long you have the GPU, minus ~15 min headroom
TOTAL_EPOCHS    = 150     # the full training length. Do NOT lower this to
                          # "fit" a session - that is what SESSION_MINUTES is
                          # for. Changing it rescales the learning-rate curve.
ABLATION        = "full"  # the main model. Others: l1_only, l1_adv, l1_adv_fft

config = {
    "model": {
        "input_channels": 1, "output_channels": 3, "base_ch": 64,
        "use_attention": True,           # CBAM attention on skip connections
        "pretrained_encoder": True,      # start from ImageNet, don't learn
                                         # "what an edge is" from scratch
        "gradient_checkpointing": False, # set True only if out of memory
        "full_res_skip": True,           # feeds the raw input to the last
                                         # layer so fine detail is recovered
                                         # rather than blurred in
        "n_scales_D": 3, "n_layers_D": 3,
    },
    "training": {
        "epochs": TOTAL_EPOCHS,
        "session_time_limit_minutes": SESSION_MINUTES,
        "batch_size": BATCH_SIZE,
        "lr_encoder": 2e-5,          # pretrained part: nudge gently
        "lr_decoder": 2e-4,          # from-scratch part: 10x faster
        "lr_discriminator": 2e-4,
        "beta1": 0.5, "beta2": 0.999,
        "warmup_epochs": 5,
        "lr_min": 1e-6,
        "gradient_clip_norm": 1.0,   # stops GAN training exploding
        "ema_decay": 0.999,          # smoothed copy of the weights; this is
                                     # what gets saved and used for results
        "mixed_precision": True,     # fp16 - roughly 2x faster
        "save_freq": 5,              # a crash costs at most 5 epochs
        "val_freq": 10,
        "seed": 42,
    },
    "loss": {
        "lambda_l1": 100.0,   # get the pixels roughly right
        "lambda_adv": 1.0,    # make it look real, not blurry
        "lambda_fft": 10.0,   # match the frequency content (texture)
        "lambda_vgg": 10.0,   # match how a pretrained network "sees" it
        "lambda_ssim": 5.0,   # match structure directly
    },
    "active_ablation": ABLATION,
    "data": {
        "dataset_type": "kaggle",     # only this dataset, not SEN1-2 as well
        "split_strategy": "scene",    # THE anti-leakage setting. Don't change.
        "sen12_root": "./data/SEN1-2",
        "train_seasons": ["spring", "summer", "fall"],
        "val_seasons": ["winter"], "test_seasons": ["winter"],
        "kaggle_root": DATA_ROOT,
        "train_terrain": ["agri", "barrenland", "grassland"],
        "val_terrain": ["urban"], "test_terrain": ["urban"],
        "image_size": 256,
        "subset_size": None,
        "num_workers": NUM_WORKERS,
    },
    "augmentation": {
        "horizontal_flip": True, "vertical_flip": True, "rotation_90": True,
        "sar_gaussian_noise": True, "eo_brightness_jitter": True,
    },
    "paths": {
        "checkpoint_dir": "./checkpoints",
        "output_dir": "./outputs",
        "log_dir": "./logs",
    },
}

CFG_PATH = "config_local.yaml"
with open(CFG_PATH, "w", encoding="utf-8") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False,
              allow_unicode=True)

print(f"Written -> {CFG_PATH}")
print(f"   dataset      : {DATA_ROOT}")
print(f"   batch_size   : {BATCH_SIZE}")
print(f"   num_workers  : {NUM_WORKERS}")
print(f"   total epochs : {TOTAL_EPOCHS}")
print(f"   this session : stops after {SESSION_MINUTES} min "
      f"({SESSION_MINUTES/60:.1f} h)")

## 6 · The leakage gate

This builds the three splits and checks that **no scene appears in more than
one**. If any does it stops, because training would produce numbers that look
good and aren't.

### What correct output looks like on this dataset

```
[Split] scene-disjoint, stratified over 4 terrain(s) | 30 scenes ->
        train=13,734 val=734 test=1,532
```

| Reading | Meaning |
|---|---|
| **~30 scenes**, 16,000 patches | correct for this archive — carry on |
| **fewer than ~10** | the filename pattern wasn't read (see cell 4); the split degrades toward terrain-based |
| **any LEAK line** | stop — do not train |

Thirty scenes is genuinely few for 16,000 patches, roughly 530 tiles per scene.
That's a property of the dataset, not a fault, and it's worth mentioning in your
report rather than leaving someone to notice it: the test set covers five
geographic scenes, so per-terrain numbers carry real variance.

### Why the splits are balanced by terrain

Scenes are allocated per terrain rather than from one pool. Allocating globally
let whichever scenes happened to land in test dominate it — on this archive that
gave 1,080 barrenland patches against 32 grassland, which would have made the
headline metric a barrenland metric. Each split now gets all four terrains in
proportion.

In [ ]:
from data.dataloader import SARtoEODataset

with open(CFG_PATH, encoding="utf-8") as f:
    audit_cfg = yaml.safe_load(f)

scenes, counts = {}, {}
for split in ("train", "val", "test"):
    ds = SARtoEODataset(audit_cfg, split=split, augment=False)
    scenes[split] = {_scene_key(p[0]) for p in ds.pairs}
    counts[split] = len(ds.pairs)

print()
print(f"{'split':<8}{'patches':>10}{'scenes':>10}")
print("-" * 28)
for s in ("train", "val", "test"):
    print(f"{s:<8}{counts[s]:>10,}{len(scenes[s]):>10,}")
print("-" * 28)
print(f"{'total':<8}{sum(counts.values()):>10,}")
print()

leaked = False
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    shared = scenes[a] & scenes[b]
    status = "OK" if not shared else f"LEAK - {len(shared)} shared scenes"
    print(f"   {a:<6} vs {b:<5} : {status}")
    leaked |= bool(shared)

if leaked:
    raise SystemExit("Splits are leaking. Do not train - results would be fake.")

print()
print("PASSED - no scene appears in two splits.")
n_scenes = sum(len(v) for v in scenes.values())
if n_scenes < 20 and sum(counts.values()) > 1000:
    print(f"NOTE: only {n_scenes} scene groups for {sum(counts.values()):,} "
          f"patches - see the warning in cell 4.")

## 7 · Smoke test

Builds the model and pushes one batch through it. Thirty seconds here saves you
discovering a shape mismatch or an out-of-memory error four hours in.

**If this raises an out-of-memory error**, go back to cell 2, lower
`BATCH_SIZE`, and re-run from there.

In [ ]:
from models.generator import UNetGenerator
from models.discriminator import MultiScaleDiscriminator

device = torch.device("cuda")
torch.cuda.reset_peak_memory_stats()

G = UNetGenerator(in_channels=1, out_channels=3, use_attention=True,
                  pretrained=True, full_res_skip=True).to(device)
D = MultiScaleDiscriminator().to(device)

with torch.no_grad(), torch.amp.autocast(device_type="cuda"):
    fake = G(torch.randn(BATCH_SIZE, 1, 256, 256, device=device))
    disc = D(torch.randn(BATCH_SIZE, 1, 256, 256, device=device),
             torch.randn(BATCH_SIZE, 3, 256, 256, device=device))

print(f"Generator output : {tuple(fake.shape)}  (want [{BATCH_SIZE}, 3, 256, 256])")
print(f"Value range      : [{fake.min():.2f}, {fake.max():.2f}]  (want about -1 to 1)")
print(f"Discriminator    : {[tuple(d.shape) for d in disc]}  (3 scales)")
print()
print(f"Generator params : {sum(p.numel() for p in G.parameters()):,}")
print(f"Peak VRAM        : {torch.cuda.max_memory_allocated()/1e9:.2f} GB "
      f"of {VRAM_GB:.1f} GB")
print()
print("Note this is inference only. Training uses roughly 3x this much,")
print("so if peak VRAM above is over a third of your card, lower BATCH_SIZE.")

del G, D, fake, disc
torch.cuda.empty_cache()

## 8 · Pilot run — 10 minutes, not 4 hours

Trains 5 epochs on 200 images. It won't learn anything useful; the point is to
prove the whole loop works — data loading, losses, validation, checkpoint
saving — before you commit a real session to it.

**Do this once, on your first day.** Skip it afterwards.

In [ ]:
import copy
pilot = copy.deepcopy(config)
pilot["training"].update(epochs=5, val_freq=5, save_freq=5,
                         warmup_epochs=1, session_time_limit_minutes=None)
pilot["data"]["subset_size"] = 200
pilot["paths"] = {"checkpoint_dir": "./_pilot/ck",
                  "output_dir": "./_pilot/out",
                  "log_dir": "./_pilot/logs"}
with open("config_pilot.yaml", "w", encoding="utf-8") as f:
    yaml.dump(pilot, f, default_flow_style=False, sort_keys=False,
              allow_unicode=True)

from train import train, load_config, make_dirs
cfg_p = load_config("config_pilot.yaml")
make_dirs(cfg_p)
_ = train(cfg_p)

print()
print("Pilot finished. If you saw per-epoch lines and a saved checkpoint,")
print("the pipeline works. Delete _pilot/ whenever you like.")

## 9 · The real training run

This is the long one. Expect roughly **2–4 minutes per epoch** on an A5000.

**What you'll see each epoch:**

```
[Epoch 003/150] G=42.1 (l1=28.3 adv=0.9 fft=3.1 vgg=8.2 ssim=1.6) D=0.51
                | lr_enc=1.2e-05 lr_dec=1.2e-04 | 9.4min elapsed | ETA 7.6h
```

- **G** — generator loss. Should trend down. Bumps are normal in GAN training.
- **D** — discriminator loss. Should hover near **0.5**. If it crashes to 0 the
  discriminator has won and the generator will stop improving.
- **ETA** — measured on your hardware. Trust this over any estimate.

**Read the ETA after epoch 2 and write it down.** Multiply by 2.85 to budget the
full four-config ablation study later.

When your time budget runs out it stops on a clean epoch boundary, saves
everything, and tells you where it got to. **Re-run this same cell next session
to continue** — it resumes automatically.

In [ ]:
cfg = load_config(CFG_PATH)
make_dirs(cfg)

print("=" * 66)
print(f" Training '{ABLATION}' -> epoch {TOTAL_EPOCHS}, "
      f"stopping after {SESSION_MINUTES} min this session")
print("=" * 66)
print()

G = train(cfg)

print()
print("Session finished. Run cell 10 to see where you are.")

## 10 · Where am I, and what do I have?

Run this after training to see progress and confirm your files exist.

In [ ]:
CK = f"checkpoints/{ABLATION}"
done = sorted(int(os.path.basename(f).replace("epoch_", "").replace(".pth", ""))
              for f in glob.glob(f"{CK}/epoch_*.pth")
              if os.path.basename(f).replace("epoch_", "").replace(".pth", "").isdigit())
last = done[-1] if done else 0

print(f"Epochs completed : {last} / {TOTAL_EPOCHS}")
if last:
    pct = 100 * last / TOTAL_EPOCHS
    print(f"Progress         : {pct:.0f}%  [{'#' * int(pct/5):<20}]")
print()
print("Checkpoints on disk:")
for f in sorted(glob.glob(f"{CK}/*.pth")):
    print(f"   {os.path.basename(f):<20} {os.path.getsize(f)/1e6:>8.1f} MB")
print()

if last >= TOTAL_EPOCHS:
    print("TRAINING COMPLETE. Go to cell 11 to evaluate.")
else:
    print(f"{TOTAL_EPOCHS - last} epochs left. Re-run cell 9 next session.")

print()
print("BEFORE YOU LEAVE THE LAB — copy these somewhere safe:")
print("   checkpoints/   outputs/   logs/")
print("Shared machines get wiped. A file in one place is a file you can lose.")

## 11 · Evaluate

Only meaningful once training has finished. Produces the numbers for your
report.

**What the metrics mean:**

| Metric | Direction | What it measures |
|---|---|---|
| SSIM | higher | structural similarity — is the layout right? |
| PSNR | higher | raw pixel error. Weakest of the four here |
| LPIPS | **lower** | perceptual distance. The one that matches human judgement best |
| FID | **lower** | do the outputs look like real satellite images as a set? |

Judge this project mainly on **LPIPS and SSIM**. PSNR rewards blurry averaging,
which is exactly the failure mode you're trying to avoid.

In [ ]:
from eval import run_inference_to_dir, evaluate_dirs

WEIGHTS = f"{CK}/best.pth"
if not os.path.exists(WEIGHTS):
    WEIGHTS = f"{CK}/final.pth"
if not os.path.exists(WEIGHTS):
    raise SystemExit(f"No checkpoint in {CK} - has training run?")

print(f"Evaluating: {WEIGHTS}")
run_inference_to_dir(CFG_PATH, WEIGHTS, "test",
                     "outputs/eval_preds", "outputs/eval_gt", use_tta=False)
metrics = evaluate_dirs("outputs/eval_preds", "outputs/eval_gt",
                        "outputs/metrics_test.csv", split="test")

print()
print("=" * 52)
print("  RESULTS - scene-disjoint test split")
print("=" * 52)
print(f"  SSIM  (higher better) : {metrics['ssim']:.4f}")
print(f"  PSNR  (higher better) : {metrics['psnr']:.2f} dB")
print(f"  LPIPS (lower  better) : {metrics['lpips']:.4f}")
print(f"  FID   (lower  better) : {metrics['fid']:.2f}")
print("=" * 52)
print()
print("No train/test overlap - these are honest numbers.")

## 12 · Look at the pictures

Numbers don't tell you whether a city looks like a city. Sample outputs are
saved during validation — this shows you the most recent ones.

Each strip is **SAR input | what the model generated | the real optical image**.

In [ ]:
from IPython.display import Image as IPyImage, display

samples = sorted(glob.glob(f"outputs/samples/{ABLATION}/*.png"))
if not samples:
    print("No samples yet - they're written during validation "
          f"(every {config['training']['val_freq']} epochs).")
else:
    print(f"{len(samples)} sample images. Showing the 3 most recent:")
    print("Layout: SAR input | generated | ground truth")
    print()
    for f in samples[-3:]:
        print(os.path.basename(f))
        display(IPyImage(filename=f))

## 13 · The ablation study

Four training runs that differ in exactly one thing each — which loss terms are
switched on. This is what tells you whether the adversarial, FFT, VGG and
MS-SSIM terms actually earn their place, rather than just assuming they do.

| Config | Losses used |
|---|---|
| `l1_only` | L1 only |
| `l1_adv` | + adversarial |
| `l1_adv_fft` | + FFT |
| `full` | + VGG + MS-SSIM |

**Don't run this from the notebook.** It takes ~21 hours across several
sessions. Use a terminal so it isn't tied to the notebook staying open:

```bash
python run_ablations.py --config config_local.yaml \
       --epochs 150 --batch-size 16 --num-workers 8
```

Run that same line each session. Finished configs are skipped automatically, so
it's safe to repeat. When all four are done it writes
`outputs/ablation_comparison.md`, ready to paste into your README.

The cell below just shows the current state.

In [ ]:
import csv
rows = []
for a in ["l1_only", "l1_adv", "l1_adv_fft", "full"]:
    p = f"outputs/metrics_{a}_test.csv"
    if os.path.exists(p):
        with open(p, encoding="utf-8") as f:
            rows.append((a, next(csv.DictReader(f))))

if not rows:
    print("No ablation results yet. Run run_ablations.py in a terminal.")
else:
    print(f"{'config':<14}{'LPIPS':>9}{'FID':>9}{'SSIM':>9}{'PSNR':>9}")
    print("-" * 50)
    for a, r in rows:
        print(f"{a:<14}{float(r['lpips']):>9.4f}{float(r['fid']):>9.1f}"
              f"{float(r['ssim']):>9.4f}{float(r['psnr']):>9.2f}")
    print("-" * 50)
    print(f"{len(rows)} of 4 configurations complete.")

## 14 · Before you close the laptop

**Every single session:**

```bash
cp -r checkpoints outputs logs /path/to/your/backup/
```

Or copy them to a USB drive or cloud folder. Lab machines get reimaged without
warning, and a checkpoint that exists in exactly one place is a checkpoint you
are about to lose.

**Next session:** open this notebook, run cells 1–7, then cell 9. It picks up
exactly where it stopped.

---

### If something goes wrong

| Symptom | What it means | Fix |
|---|---|---|
| `CUDA out of memory` | batch too big for the card | lower `BATCH_SIZE` in cell 2, re-run from there |
| Loss becomes `nan` | training diverged | reduce `lr_decoder` to `1e-4`, delete checkpoints, restart |
| `D` loss drops to ~0 | discriminator overpowered the generator | lower `lambda_adv` to `0.5` |
| Very slow epochs | CPU can't feed the GPU | raise `NUM_WORKERS` in cell 2 |
| Splits look tiny | dataset didn't fully unzip | re-check cell 3's pair counts |

### Once Phase 1 is solid

Phase 2 is a diffusion model — usually better texture, much slower. Only worth
starting when these numbers look reasonable and you've written up Phase 1.